# AFIR for a molecule on a surface with NVIDIA ALCHEMI, on a platform GPU

Places a molecule at random over a slab many times and relaxes every placement with MACE through the
[NVIDIA ALCHEMI toolkit](https://github.com/NVIDIA/nvalchemi-toolkit), all placements batched on one GPU: first
under the artificial force (AFIR) between molecule and slab, then without it. The final structures are ranked by
energy, and the lowest are saved as materials.

The workflow has three steps, each in its own Python environment: *Configurations* builds the placements from the
job's two materials with made, *AFIR (ALCHEMI)* relaxes them, and *Final structures* saves the lowest as the job's
final structures. This notebook builds the slab, fetches the molecule, saves both as materials, creates the workflow,
runs one job on the two materials and shows the ranked configurations and the lowest final structures.

Outside the platform's JupyterLite, use Python 3.10 or 3.11 and install the packages once with `pip install "mat3ra-notebooks-utils[all]"`.

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples")

### 1.2. Set parameters

In [ ]:
from datetime import datetime

from mat3ra.ide.compute import QueueName

# 2. Auth and organization parameters
ORGANIZATION_NAME = None  # set to use an organization as the owner, otherwise the personal account

# 3. Material parameters
CRYSTAL_NAME = "Cu"  # from Materials Standata
MILLER_INDICES = (1, 1, 1)
NUMBER_OF_LAYERS = 2
XY_SUPERCELL_MATRIX = [[4, 0], [0, 4]]
USE_CONVENTIONAL_CELL = False  # the supercell repeats the primitive surface cell
VACUUM = 15.0  # in angstroms
MOLECULE_NAME = "water"  # from PubChem

# 4. Workflow parameters
NUMBER_OF_CONFIGURATIONS = 100  # random placements of the molecule over the slab
GAMMA = 100  # kJ/mol, model collision energy of the AFIR force
RANDOM_SEED = 0  # of the random placements
NUMBER_OF_FINAL_STRUCTURES = 5  # the lowest-energy final structures kept and saved as materials
PYTHON_VERSION = "3.11.10"
PYTHON_BUILD = "Default"

# 5. Compute parameters
CLUSTER_NAME = "cluster-003"
QUEUE_NAME = QueueName.GOF  # H100 nodes on cluster-003 (GOF on-demand, GSF spot)
TIME_LIMIT = "01:00:00"

# 6. Job parameters
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 30  # seconds

## 2. Authenticate and initialize API client
### 2.1. Authenticate

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate


await authenticate()

### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()

### 2.3. Select account

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Create materials
### 3.1. Build the slab

In [ ]:
from mat3ra.made.material import Material
from mat3ra.made.tools.helpers import create_slab
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize
from mat3ra.standata.materials import Materials

crystal = Material.create(Materials.get_by_name_first_match(CRYSTAL_NAME))
slab = create_slab(
    crystal,
    miller_indices=MILLER_INDICES,
    number_of_layers=NUMBER_OF_LAYERS,
    vacuum=VACUUM,
    xy_supercell_matrix=XY_SUPERCELL_MATRIX,
    use_conventional_cell=USE_CONVENTIONAL_CELL,
)
visualize(slab)

### 3.2. Fetch the molecule from PubChem

In [ ]:
import io
from urllib.parse import quote

from ase.io import read
from mat3ra.made.tools.convert import from_ase
from mat3ra.notebooks_utils.io import read_from_url

PUBCHEM_STRUCTURE_URL = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{name}/SDF?record_type=3d"

molecule_sdf = await read_from_url(PUBCHEM_STRUCTURE_URL.format(name=quote(MOLECULE_NAME)))
molecule = Material.create(from_ase(read(io.StringIO(molecule_sdf), format="sdf")))
molecule.name = MOLECULE_NAME
visualize(molecule)

### 3.3. Save materials to the platform

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

saved_materials = [get_or_create_material(client, material, ACCOUNT_ID) for material in (slab, molecule)]

## 4. Create the workflow
### 4.1. Step 1: configurations

Places the molecule `NUMBER_OF_CONFIGURATIONS` times over the slab, each at a random orientation and a random point
of the surface cell, with its lowest atom above the top slab atom by the sum of their covalent radii plus 0.5 Å, as
GRRM's random initial structures [1]. Slab atoms are labelled 0, molecule atoms 1, and each configuration
keeps its rotation and position in its metadata.

In [ ]:
CONFIGURATIONS_SCRIPT = r'''# ---------------------------------------------------------------- #
#                                                                  #
#  Configurations step of the AFIR molecule-on-surface workflow.   #
#                                                                  #
#  Places the job's molecule over its slab                         #
#  NUMBER_OF_CONFIGURATIONS times with made, each at a random      #
#  orientation and a random point of the surface cell, as          #
#  GRRM17's random initial structures [1].                         #
#                                                                  #
#  Reads:  the job's two materials, written into this script by    #
#          the platform when the job is created; the non-periodic  #
#          one is the molecule.                                    #
#  Writes: CONFIGURATIONS to the job's global scope: the placed    #
#          structures as materials, slab atoms labelled 0 and      #
#          molecule atoms 1, each with its rotation and fractional #
#          position in metadata.placement. It is the only output   #
#          of the script.                                          #
#                                                                  #
#  [1] S. Maeda et al., J. Comput. Chem. 39, 233 (2018).           #
#                                                                  #
# ---------------------------------------------------------------- #

import json

import numpy as np
from ase.data import atomic_numbers, covalent_radii
from mat3ra.made.material import Material
from mat3ra.made.tools.convert.interface_parts_enum import InterfacePartsEnum
from mat3ra.made.tools.modify import translate_by_vector
from mat3ra.made.tools.operations.core.binary import merge
from mat3ra.made.tools.operations.core.unary import rotate

NUMBER_OF_CONFIGURATIONS = 100
RANDOM_SEED = 0
CONTACT_OFFSET = 0.5  # Å, lowest molecule atom at R_molecule + R_slab + CONTACT_OFFSET above the top slab atom
# The job's materials, written here by the platform when the job is created.
MATERIALS = json.loads(r"""{{ materials | dump | safe }}""")


def extreme_atom_height_and_radius(material, extremum):
    """
    Returns the height and the covalent radius of the highest or the lowest atom of a material.

    Args:
        material (Material): The material to search.
        extremum (Callable): max for the highest atom, min for the lowest.

    Returns:
        Tuple[float, float]: The atom's cartesian z and its covalent radius, both in Å.
    """
    cartesian = material.clone()
    cartesian.to_cartesian()
    height, element = extremum(zip([coordinate[2] for coordinate in cartesian.basis.coordinates.values],
                                   cartesian.basis.elements.values))
    return height, covalent_radii[atomic_numbers[element]]


def create_configuration(slab, molecule, top_atom_height_and_radius, random_generator, index):
    """
    Places the molecule once over the slab, as a random initial structure of GRRM17's MC-AFIR.

    The orientation is uniformly random (a normalised Gaussian quaternion), the molecule's center goes to a random
    point of the surface cell, and its lowest atom sits R_molecule + R_slab + CONTACT_OFFSET above the top slab atom,
    the RandDC rule of GRRM17 (S. Maeda et al., J. Comput. Chem. 39, 233 (2018)).

    Args:
        slab (Material): The slab, in crystal units, labelled 0.
        molecule (Material): The molecule, in crystal units.
        top_atom_height_and_radius (Tuple[float, float]): Height and covalent radius of the top slab atom, in Å.
        random_generator (numpy.random.Generator): The seeded generator of all placements.
        index (int): The configuration's index, used in its name.

    Returns:
        MaterialWithBuildMetadata: The slab and the placed molecule, molecule atoms labelled 1, with the rotation and
        the fractional position in metadata.placement.
    """
    quaternion = random_generator.normal(size=4)
    axis = (quaternion[1:] / np.linalg.norm(quaternion[1:])).tolist()
    angle = float(2 * np.degrees(np.arccos(quaternion[0] / np.linalg.norm(quaternion))))
    rotated = rotate(molecule, axis=axis, angle=angle, wrap=False)
    rotated.set_lattice(slab.lattice)
    top_height, top_radius = top_atom_height_and_radius
    lowest_height, lowest_radius = extreme_atom_height_and_radius(rotated, min)
    surface_fractions = [random_generator.random(), random_generator.random(), 0]
    surface_point = slab.basis.cell.convert_point_to_cartesian(surface_fractions)
    molecule_cell_center = molecule.basis.cell.convert_point_to_cartesian([0.5, 0.5, 0.5])
    vertical_shift = top_height + top_radius + lowest_radius + CONTACT_OFFSET - lowest_height
    placed = translate_by_vector(rotated, vector=[surface_point[0] - molecule_cell_center[0],
                                                  surface_point[1] - molecule_cell_center[1], vertical_shift],
                                 use_cartesian_coordinates=True)
    placed.set_labels_from_value(InterfacePartsEnum.FILM.value)
    configuration = merge([slab, placed], material_name=f"{molecule.name} on {slab.name}, configuration {index}")
    configuration.metadata.placement = {"rotation_axis": axis, "rotation_angle_deg": angle,
                                        "position_u": surface_fractions[0], "position_v": surface_fractions[1]}
    return configuration


def main():
    """
    Builds NUMBER_OF_CONFIGURATIONS configurations from the job's materials and prints them as CONFIGURATIONS.
    """
    slab = Material.create(next(material_config for material_config in MATERIALS
                                if not material_config.get("isNonPeriodic")))
    molecule = Material.create(next(material_config for material_config in MATERIALS
                                    if material_config.get("isNonPeriodic")))
    for material in (slab, molecule):
        material.to_crystal()
    slab.set_labels_from_value(InterfacePartsEnum.SUBSTRATE.value)
    top_atom_height_and_radius = extreme_atom_height_and_radius(slab, max)
    random_generator = np.random.default_rng(RANDOM_SEED)
    configurations = [
        create_configuration(slab, molecule, top_atom_height_and_radius, random_generator, index).to_dict()
        for index in range(NUMBER_OF_CONFIGURATIONS)
    ]
    print(json.dumps({"name": "CONFIGURATIONS", "value": configurations, "scope": "global"}))


if __name__ == "__main__":
    main()
'''

### 4.2. Step 2: AFIR with ALCHEMI

Relaxes all configurations together with the slab fixed [3], first under the AFIR force [1] of strength `GAMMA`
(100 kJ/mol with random orientations [2]) between molecule and slab, then without it. It writes `results.csv` with
every configuration ranked by final energy, its starting orientation and the fragments its molecule ends in;
`final_structures.extxyz` with the `NUMBER_OF_FINAL_STRUCTURES` lowest; and `energies.png`, the final energy of each
configuration. Another backend or another ALCHEMI version replaces this script and its packages; steps 1 and 3 stay.

In [ ]:
AFIR_SCRIPT = r'''# ---------------------------------------------------------------- #
#                                                                  #
#  AFIR step of the AFIR molecule-on-surface workflow, with MACE   #
#  through NVIDIA ALCHEMI on one GPU.                              #
#                                                                  #
#  Relaxes all configurations as one batch with the slab fixed,    #
#  first under the AFIR force between molecule and slab (GRRM17    #
#  Eqs. 1-3 [1], strength GAMMA), then without it, and ranks the   #
#  final structures by energy.                                     #
#                                                                  #
#  Reads:  CONFIGURATIONS from the job's global scope, written     #
#          into this script by the compute node before it runs.    #
#  Writes: results.csv, every configuration ranked by final        #
#          energy, with its starting rotation and position and the #
#          number of fragments its molecule ends in;               #
#          final_structures.extxyz, the NUMBER_OF_FINAL_STRUCTURES #
#          lowest in that order; and energies.png, final energy    #
#          against configuration index. All three are results of   #
#          this unit.                                              #
#                                                                  #
#  [1] S. Maeda et al., J. Comput. Chem. 39, 233 (2018).           #
#                                                                  #
# ---------------------------------------------------------------- #

import csv
import json

import matplotlib
import numpy as np
import torch
from ase.build import separate
from ase.calculators.singlepoint import SinglePointCalculator
from ase.data import covalent_radii
from ase.io import write
from ase.units import kJ, mol
from loguru import logger
from mat3ra.made.material import Material
from mat3ra.made.tools.convert import to_ase
from mat3ra.made.tools.convert.interface_parts_enum import InterfacePartsEnum
from nvalchemi._typing import AtomCategory
from nvalchemi.data import AtomicData, Batch
from nvalchemi.dynamics import FIRE2, FusedStage
from nvalchemi.dynamics.base import ConvergenceHook, DynamicsStage
from nvalchemi.dynamics.hooks import FreezeAtomsHook
from nvalchemi.hooks import BiasedPotentialHook
from nvalchemi.models.mace import MACEWrapper

matplotlib.use("Agg")
import matplotlib.pyplot as plt

DEVICE = "cuda:0"
CHECKPOINT = "medium-mpa-0"  # MACE-MPA-0, medium
DTYPE = torch.float32
FMAX = 0.03  # eV/Å, largest atomic force at convergence
MAX_STEPS = 500  # FIRE2 steps per relaxation at most
TIME_STEP = 0.5  # initial FIRE2 step, in ALCHEMI's internal time unit (10.18 fs)
GAMMA = 100  # kJ/mol, model collision energy of the AFIR force
NUMBER_OF_FINAL_STRUCTURES = 5  # the lowest final structures kept in final_structures.extxyz
BOND_TOLERANCE = 1.2  # two molecule atoms are bonded within BOND_TOLERANCE * (R_i + R_j)
AFIR_R0 = 3.8164  # Å, GRRM17 Eq. (3)
AFIR_EPSILON = 1.0061  # kJ/mol, GRRM17 Eq. (3)
CONFIGURATIONS = json.loads(r"""{{ CONFIGURATIONS | tojson }}""")

ALPHA = GAMMA * kJ / mol / ((2 ** (-1 / 6) - (1 + (1 + GAMMA / AFIR_EPSILON) ** 0.5) ** (-1 / 6)) * AFIR_R0)  # Eq. (3)
CATEGORY_BY_PART = {InterfacePartsEnum.SUBSTRATE.value: AtomCategory.SURFACE.value,
                    InterfacePartsEnum.FILM.value: AtomCategory.GAS.value}
IN_PLANE_MASK = torch.tensor([1.0, 1.0, 0.0], device=DEVICE)
IN_PLANE_SHIFTS = torch.tensor([[a, b, 0.0] for a in (-1, 0, 1) for b in (-1, 0, 1)], device=DEVICE)
COVALENT_RADII = torch.tensor(covalent_radii, device=DEVICE, dtype=DTYPE)


def in_plane_minimum_image(vectors, cells):
    """
    Returns the shortest periodic image of each vector in the surface plane, a and b; there is no image along c.

    Args:
        vectors (torch.Tensor): Cartesian vectors, [P, 3], in Å.
        cells (torch.Tensor): The lattice vectors of each vector's structure, [P, 3, 3], rows a, b, c.

    Returns:
        torch.Tensor: The minimum-image vectors, [P, 3], in Å.
    """
    fractional = torch.einsum("pi,pij->pj", vectors, torch.linalg.inv(cells))
    fractional = fractional - torch.round(fractional) * IN_PLANE_MASK
    images = torch.einsum("psi,pij->psj", fractional.unsqueeze(1) + IN_PLANE_SHIFTS, cells)
    nearest = images.norm(dim=-1).argmin(dim=1)
    return torch.take_along_dim(images, nearest[:, None, None], dim=1).squeeze(1)


def afir_energy(positions, batch):
    """
    Returns the AFIR term of every structure in the batch, between its molecule and its slab atoms.

    GRRM17 Eq. (1), α Σ ω_ij r_ij / Σ ω_ij over molecule atoms i and slab atoms j, with ω_ij = ((R_i + R_j) / r_ij)^6
    (Eq. 2), R the covalent radii and r_ij at the minimum image in the surface plane; α is ALPHA, Eq. (3)
    (S. Maeda et al., J. Comput. Chem. 39, 233 (2018)).

    Args:
        positions (torch.Tensor): Atomic positions of the batch, [V, 3], in Å.
        batch (Batch): The batch, with atom_categories GAS for the molecule and SURFACE for the slab.

    Returns:
        torch.Tensor: The AFIR term of each structure, [B], in eV.
    """
    molecule_atoms = torch.where(batch.atom_categories == AtomCategory.GAS.value)[0]
    surface_atoms = torch.where(batch.atom_categories == AtomCategory.SURFACE.value)[0]
    molecule_index, surface_index = torch.where(
        batch.batch_idx[molecule_atoms][:, None] == batch.batch_idx[surface_atoms][None, :])
    pair_molecule_atoms, pair_surface_atoms = molecule_atoms[molecule_index], surface_atoms[surface_index]
    pair_structures = batch.batch_idx[pair_molecule_atoms]
    distances = in_plane_minimum_image(positions[pair_molecule_atoms] - positions[pair_surface_atoms],
                                       batch.cell[pair_structures]).norm(dim=-1)
    radii = (COVALENT_RADII[batch.atomic_numbers[pair_molecule_atoms]]
             + COVALENT_RADII[batch.atomic_numbers[pair_surface_atoms]])
    weights = (radii / distances) ** 6
    numerator = torch.zeros(batch.num_graphs, device=positions.device, dtype=positions.dtype)
    denominator = torch.zeros_like(numerator)
    return ALPHA * (numerator.index_add(0, pair_structures, weights * distances)
                    / denominator.index_add(0, pair_structures, weights))


def afir_bias(batch):
    """
    Returns the AFIR energy and forces of the batch, as BiasedPotentialHook expects them.

    Args:
        batch (Batch): The batch after the model's forward pass.

    Returns:
        Tuple[torch.Tensor, torch.Tensor]: Energy per structure, [B, 1], in eV, and force per atom, [V, 3],
        in eV/Å.
    """
    positions = batch.positions.clone().requires_grad_(True)
    energy = afir_energy(positions, batch)
    forces = -torch.autograd.grad(energy.sum(), positions)[0]
    return energy.detach().unsqueeze(-1), forces


def atomic_data(atoms):
    """
    Converts a configuration to ALCHEMI's AtomicData, its labels to atom categories: 0 to SURFACE, 1 to GAS.

    Args:
        atoms (ase.Atoms): The configuration, slab atoms tagged 0 and molecule atoms 1.

    Returns:
        AtomicData: The configuration on DEVICE, with zeroed forces and energy.
    """
    data = AtomicData.from_atoms(atoms, device=DEVICE, dtype=DTYPE)
    data["atom_categories"] = torch.tensor([CATEGORY_BY_PART[tag] for tag in atoms.get_tags()], device=DEVICE)
    data["forces"] = torch.zeros_like(data.positions)
    data["energy"] = torch.zeros((1, 1), device=DEVICE, dtype=DTYPE)
    return data


def relaxed_atoms(original, relaxed_data):
    """
    Returns a copy of a structure at its relaxed positions, with its energy and forces.

    Args:
        original (ase.Atoms): The structure before the relaxation.
        relaxed_data (AtomicData): The same structure after it.

    Returns:
        ase.Atoms: The relaxed structure, energy and forces on a SinglePointCalculator.
    """
    atoms = original.copy()
    atoms.positions = relaxed_data.positions.cpu().numpy()
    atoms.calc = SinglePointCalculator(atoms, energy=float(relaxed_data.energy.item()),
                                       forces=relaxed_data.forces.cpu().numpy())
    return atoms


def relax(ase_structures, model, bias_hooks):
    """
    Relaxes all structures as one batch with fixed-cell FIRE2, slab atoms frozen.

    Args:
        ase_structures (List[ase.Atoms]): The structures, slab atoms tagged 0 and molecule atoms 1.
        model (MACEWrapper): The MACE model.
        bias_hooks (List[BiasedPotentialHook]): Bias potentials added after each forward pass; [] for none.

    Returns:
        List[ase.Atoms]: The relaxed structures, in the input order.
    """
    optimizer = FIRE2(model=model, dt=TIME_STEP, n_steps=MAX_STEPS, convergence_hook=ConvergenceHook(
        criteria=[{"key": "forces", "threshold": FMAX, "reduce_op": "norm", "reduce_dims": -1}]))
    optimizer.register_hook(FreezeAtomsHook(freeze_category=AtomCategory.SURFACE.value))
    stage = FusedStage(sub_stages=[(0, optimizer)])
    for hook in model.make_neighbor_hooks():
        stage.register_hook(hook, stage=DynamicsStage.BEFORE_COMPUTE)
    for hook in bias_hooks:
        stage.register_hook(hook, stage=DynamicsStage.AFTER_COMPUTE)
    batch = stage.run(Batch.from_data_list([atomic_data(atoms) for atoms in ase_structures]))
    return [relaxed_atoms(atoms, relaxed_data) for atoms, relaxed_data in zip(ase_structures, batch.to_data_list())]


def molecule_fragments(atoms):
    """
    Returns the number of fragments the molecule of a structure is in: 1 if intact, more if it split.

    Args:
        atoms (ase.Atoms): The structure, molecule atoms tagged 1.

    Returns:
        int: The number of connected groups of molecule atoms, bonded within BOND_TOLERANCE times their covalent radii.
    """
    return len(separate(atoms[atoms.get_tags() == InterfacePartsEnum.FILM.value], scale=BOND_TOLERANCE))


def energy_and_max_force(atoms):
    """
    Returns the energy of a structure, in eV, and the largest force on any of its atoms, in eV/Å.
    """
    return atoms.get_potential_energy(), np.linalg.norm(atoms.get_forces(), axis=1).max()


def plot_energies(final_structures, lowest_indices):
    """
    Plots the final energy of every configuration against its index, the kept lowest marked, to energies.png.

    Args:
        final_structures (List[ase.Atoms]): The structures after the relaxation without the AFIR force.
        lowest_indices (List[int]): The indices of the kept lowest structures.
    """
    energies = [atoms.get_potential_energy() for atoms in final_structures]
    figure, axes = plt.subplots()
    axes.scatter(range(len(energies)), energies, s=12, label="all configurations")
    axes.scatter(lowest_indices, [energies[index] for index in lowest_indices], s=40, color="tab:red",
                 label=f"{len(lowest_indices)} lowest, kept")
    axes.set(xlabel="Configuration index", ylabel="Final energy (eV)")
    axes.ticklabel_format(axis="y", useOffset=False)
    axes.legend()
    figure.savefig("energies.png", dpi=150, bbox_inches="tight")


def write_results(afir_structures, final_structures):
    """
    Ranks the configurations by final energy, writes all of them to results.csv with the placement each started from
    (atoms.info["metadata"]["placement"]) and its molecule's fragments, the NUMBER_OF_FINAL_STRUCTURES lowest to
    final_structures.extxyz in that order, and the energies to energies.png.

    Args:
        afir_structures (List[ase.Atoms]): The structures after the relaxation under the AFIR force.
        final_structures (List[ase.Atoms]): The structures after the relaxation without it.
    """
    order = np.argsort([atoms.get_potential_energy() for atoms in final_structures])
    with open("results.csv", "w", newline="") as results_file:
        writer = csv.writer(results_file)
        writer.writerow(["configuration", "rank", "afir_function_eV", "afir_max_force_eV_per_A", "energy_eV",
                         "max_force_eV_per_A", "rotation_axis_x", "rotation_axis_y", "rotation_axis_z",
                         "rotation_angle_deg", "position_u", "position_v", "molecule_fragments"])
        for rank, index in enumerate(order, start=1):
            placement = final_structures[index].info["metadata"]["placement"]
            writer.writerow([index, rank, *energy_and_max_force(afir_structures[index]),
                             *energy_and_max_force(final_structures[index]), *placement["rotation_axis"],
                             placement["rotation_angle_deg"], placement["position_u"], placement["position_v"],
                             molecule_fragments(final_structures[index])])
            final_structures[index].info.update(configuration=int(index), rank=rank)
    lowest_indices = order[:NUMBER_OF_FINAL_STRUCTURES].tolist()
    write("final_structures.extxyz", [final_structures[index] for index in lowest_indices])
    plot_energies(final_structures, lowest_indices)


def main():
    """
    Relaxes CONFIGURATIONS under the AFIR force, then without it, and writes the ranked results.
    """
    logger.info("AFIR force α, γ = {} kJ/mol: {:.3f} eV/Å", GAMMA, ALPHA)
    ase_structures = [to_ase(Material.create(configuration)) for configuration in CONFIGURATIONS]
    model = MACEWrapper.from_checkpoint(checkpoint_path=CHECKPOINT, device=torch.device(DEVICE), dtype=DTYPE)
    afir_structures = relax(ase_structures, model, [BiasedPotentialHook(bias_fn=afir_bias)])
    final_structures = relax(afir_structures, model, [])
    write_results(afir_structures, final_structures)


if __name__ == "__main__":
    main()
'''

In [ ]:
MADE_REQUIREMENTS = "mat3ra-made[tools]==2026.10.10.post0\n"
AFIR_REQUIREMENTS = """--find-links https://download.pytorch.org/whl/cu128/torch/
--find-links https://download.pytorch.org/whl/cu128/torchvision/
--find-links https://download.pytorch.org/whl/cu128/torchaudio/
--find-links https://github.com/NVIDIA/warp/releases/expanded_assets/v1.18.0
torch==2.11.0+cu128
torchvision==0.26.0+cu128
torchaudio==2.11.0+cu128
warp-lang==1.18.0+cu12
nvalchemi-toolkit[mace,ase]==0.2.0
""" + MADE_REQUIREMENTS

### 4.3. Step 3: final structures

Takes the lowest final structures, each with the configuration it started from, and saves them as the job's final
structures: each is a material, shown in the Results tab of the AFIR step.

In [ ]:
FINAL_STRUCTURES_SCRIPT = r'''# ---------------------------------------------------------------- #
#                                                                  #
#  Final structures step of the AFIR molecule-on-surface workflow. #
#                                                                  #
#  Takes the lowest final structures of the AFIR step, each with   #
#  the configuration it started from, and makes one request body   #
#  per structure for the platform's final-structure endpoint,      #
#  which the io unit after this one sends in a loop.               #
#                                                                  #
#  Reads:  final_structures.extxyz in the job's work folder, and   #
#          CONFIGURATIONS from the job's global scope, written     #
#          into this script by the compute node before it runs.    #
#  Writes: FINAL_STRUCTURE_REQUESTS to the job's global scope, as  #
#          the only output of the script.                          #
#                                                                  #
# ---------------------------------------------------------------- #

import json

from ase.io import read
from mat3ra.made.material import Material
from mat3ra.made.tools.convert import from_ase

AFIR_UNIT_FLOWCHART_ID = ""
CONFIGURATIONS = json.loads(r"""{{ CONFIGURATIONS | tojson }}""")


def request_body(rank_index, atoms):
    """
    Returns the body of one final-structure request, escaped to sit inside the next io unit's JSON string.

    The platform keeps one final_structure per unit and repetition, so each structure gets its own repetition.

    Args:
        rank_index (int): The structure's rank by final energy, from 0.
        atoms (ase.Atoms): The final structure, with its configuration's index in atoms.info.

    Returns:
        str: {"jobId", "unitId", "structures": [{"initial", "final"}]} as JSON, escaped for a JSON string.
    """
    final = {**Material.create(from_ase(atoms)).to_dict(), "repetition": rank_index}
    initial = CONFIGURATIONS[atoms.info["configuration"]]
    body = {"jobId": "{{ JOB_ID }}", "unitId": AFIR_UNIT_FLOWCHART_ID,
            "structures": [{"initial": initial, "final": final}]}
    return json.dumps(json.dumps(body))[1:-1]


def main():
    """
    Reads the lowest final structures and prints their request bodies as FINAL_STRUCTURE_REQUESTS.
    """
    final_structures = read("{{ JOB_WORK_DIR }}/final_structures.extxyz", index=":")
    request_bodies = [request_body(rank_index, atoms) for rank_index, atoms in enumerate(final_structures)]
    print(json.dumps({"name": "FINAL_STRUCTURE_REQUESTS", "value": request_bodies, "scope": "global"}))


if __name__ == "__main__":
    main()
'''

### 4.4. Configure the steps

When the job is created, the platform writes the job's materials into step 1's script (`materials`; the
non-periodic one is the molecule). Step 1 prints `CONFIGURATIONS` to the job's scope, and the compute node writes
them into the scripts of steps 2 and 3 before they run. Each step's files are named after it.

In [ ]:
import copy
import re

from mat3ra.notebooks_utils.core.entity.workflow.api import set_execution_unit_input
from mat3ra.standata.workflows import WorkflowStandata


def set_constants(script, constants):
    """Sets each named constant of a script to its value."""
    for name, value in constants.items():
        script = re.sub(rf"^({name} = )\S+", rf"\g<1>{value}", script, flags=re.M)
    return script


def configure_python_step(subworkflow, template_unit_name, subworkflow_name, unit_name, script, requirements, results,
                          is_rendered_at_run_time):
    """
    Makes a Standata python subworkflow one step of the workflow.

    Args:
        subworkflow (dict): The subworkflow config, edited in place.
        template_unit_name (str): The name of the template's python unit.
        subworkflow_name (str): The step's name in the workflow.
        unit_name (str): The python unit's name; its script and requirements files are named after it.
        script (str): The unit's script.
        requirements (str): The packages installed for the unit, as a requirements.txt.
        results (list): The results the unit declares.
        is_rendered_at_run_time (bool): True to leave the script's placeholders to the compute node, which fills them
            from the job's scope when the step runs; False to let the platform fill them, with the job's materials,
            when the job is created.
    """
    unit = next(unit for unit in subworkflow["units"] if unit["name"] == template_unit_name)
    unit.update(name=unit_name, head=True, results=results)
    for application in (subworkflow["application"], unit["application"]):
        application.update(version=PYTHON_VERSION, build=PYTHON_BUILD)
    set_execution_unit_input(unit, "script.py", script)
    set_execution_unit_input(unit, "requirements.txt", requirements)
    inputs = {unit_input["template"]["name"]: unit_input for unit_input in unit["input"]}
    inputs["script.py"]["isManuallyChanged"] = is_rendered_at_run_time
    inputs["script.py"]["template"]["name"] = f"{unit_name}.py"
    inputs["requirements.txt"]["template"]["name"] = f"{unit_name}_requirements.txt"
    subworkflow.update(name=subworkflow_name, units=[unit])


python_workflows = WorkflowStandata.filter_by_application("python")
workflow_config = copy.deepcopy(python_workflows.get_by_name_first_match("custom_script.json"))
workflow_config.update(name="AFIR, molecule on surface", isMultiMaterial=True)
workflow_config["units"][0]["name"] = "Configurations"
python_script_subworkflow = python_workflows.get_by_name_first_match("python_script.json")["subworkflows"][0]
afir_subworkflow_config = copy.deepcopy(python_script_subworkflow)
final_structures_subworkflow_config = copy.deepcopy(python_script_subworkflow)
final_structures_subworkflow_config.pop("_id")
next(unit for unit in final_structures_subworkflow_config["units"] if unit["name"] == "python").pop("flowchartId")
afir_unit = next(unit for unit in afir_subworkflow_config["units"] if unit["name"] == "python")
afir_unit_flowchart_id = afir_unit["flowchartId"]
number_of_saved_structures = min(NUMBER_OF_FINAL_STRUCTURES, NUMBER_OF_CONFIGURATIONS)

configure_python_step(
    workflow_config["subworkflows"][0], template_unit_name="custom_script", subworkflow_name="Configurations",
    unit_name="configurations",
    script=set_constants(CONFIGURATIONS_SCRIPT, {"NUMBER_OF_CONFIGURATIONS": NUMBER_OF_CONFIGURATIONS,
                                                 "RANDOM_SEED": RANDOM_SEED}),
    requirements=MADE_REQUIREMENTS, results=[], is_rendered_at_run_time=False,
)
configure_python_step(
    afir_subworkflow_config, template_unit_name="python", subworkflow_name="AFIR (ALCHEMI)", unit_name="afir",
    script=set_constants(AFIR_SCRIPT, {"GAMMA": GAMMA, "NUMBER_OF_FINAL_STRUCTURES": number_of_saved_structures}),
    requirements=AFIR_REQUIREMENTS,
    results=[
        {"name": "file_content", "filetype": "csv", "basename": "results.csv"},
        {"name": "final_structure"},
        {"name": "file_content", "filetype": "text", "basename": "final_structures.extxyz"},
        {"name": "file_content", "filetype": "image", "basename": "energies.png"},
    ],
    is_rendered_at_run_time=True,
)
configure_python_step(
    final_structures_subworkflow_config, template_unit_name="python", subworkflow_name="Final structures",
    unit_name="final_structures",
    script=set_constants(FINAL_STRUCTURES_SCRIPT, {"AFIR_UNIT_FLOWCHART_ID": f'"{afir_unit_flowchart_id}"'}),
    requirements=MADE_REQUIREMENTS, results=[], is_rendered_at_run_time=True,
)

### 4.5. Create and save the workflow

Step 3 ends with a loop that saves one final structure per pass, with its starting configuration as the initial
structure.

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow
from mat3ra.wode import Subworkflow, Workflow
from mat3ra.wode.units import Unit

endpoint_options = {"method": "POST", "headers": {"Content-Type": "application/json"},
                    "data": "{{ FINAL_STRUCTURE_REQUESTS[INDEX] }}"}
save_unit = Unit(type="io", subtype="output", source="api", enableRender=True, name="save-final-structure",
                 input=[{"type": "api", "endpoint": "structures", "endpoint_options": endpoint_options,
                         "name": "FINAL_STRUCTURE"}])
exit_unit = Unit(type="assignment", name="final-structures-saved", operand="NUMBER_OF_SAVED_STRUCTURES", value="INDEX",
                 input=[])
loop_units = [
    Unit(type="assignment", name="init-index", operand="INDEX", value=0, input=[]),
    save_unit,
    Unit(type="assignment", name="increment-index", operand="INDEX", value="INDEX + 1", input=[]),
    Unit(**{"type": "condition", "name": "check-index", "statement": f"INDEX >= {number_of_saved_structures}",
            "then": exit_unit.flowchartId, "else": save_unit.flowchartId,
            "maxOccurrences": number_of_saved_structures, "input": [{"name": "INDEX", "scope": "global"}]}),
    exit_unit,
]
final_structures_subworkflow = Subworkflow(**final_structures_subworkflow_config)
for unit in loop_units:
    final_structures_subworkflow.add_unit(unit)
workflow = Workflow.create(workflow_config)
workflow.add_subworkflow(Subworkflow(**afir_subworkflow_config))
workflow.add_subworkflow(final_structures_subworkflow)
visualize_workflow(workflow)

saved_workflow = client.workflows.create(workflow.to_dict_without_special_keys(), owner_id=ACCOUNT_ID)
print(f"✅ Workflow saved to the collection: {saved_workflow['_id']}")

## 5. Create the compute configuration
### 5.1. Get list of clusters

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[cluster['hostname'] for cluster in clusters]}")

### 5.2. Create compute configuration for the job

In [ ]:
from mat3ra.ide.compute import Compute

cluster = next(cluster for cluster in clusters if CLUSTER_NAME in cluster["hostname"])
compute = Compute(cluster=cluster, queue=QUEUE_NAME, timeLimit=TIME_LIMIT)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, nodes: {compute.nodes}, ppn: {compute.ppn}, time limit: {TIME_LIMIT}")

## 6. Create the job

In [ ]:
from mat3ra.notebooks_utils.job import create_job

job = create_job(
    api_client=client,
    materials=saved_materials,
    workflow=workflow,
    project_id=project_id,
    owner_id=ACCOUNT_ID,
    prefix=f"{workflow.name}: {MOLECULE_NAME} on {slab.name}, {NUMBER_OF_CONFIGURATIONS} configurations {timestamp}",
    compute=compute.to_dict(),
)
job_id = job["_id"]
print(f"✅ Created job: {job_id}")

## 7. Submit the job and monitor the status

In [ ]:
from mat3ra.notebooks_utils.api.job import submit_jobs

submit_jobs(client.jobs, [job_id])

In [ ]:
from mat3ra.notebooks_utils.api.job import wait_for_jobs_to_finish_async

await wait_for_jobs_to_finish_async(client.jobs, [job_id], poll_interval=POLL_INTERVAL)

## 8. Results
### 8.1. Configurations ranked by final energy

`afir_function_eV` is the MACE energy plus the AFIR term at the end of the first relaxation, `energy_eV` the MACE
energy of the final structure; the `max_force` columns give the largest force on an atom at the end of each. The
`rotation_*` columns give the rotation applied to the molecule about its center, `position_u` and `position_v` the
fractional point of the surface cell it was placed over, and `molecule_fragments` how many pieces the molecule
ends in: 1 if intact, more if it split.

In [ ]:
import pandas as pd

file_url_by_name = {file["key"].rsplit("/", 1)[-1]: file["signedUrl"] for file in client.jobs.list_files(job_id)}
pd.read_csv(io.StringIO(await read_from_url(file_url_by_name["results.csv"])))

### 8.2. Final energy of each configuration

In [ ]:
from IPython.display import Image

Image(data=await read_from_url(file_url_by_name["energies.png"], as_bytes=True))

### 8.3. The lowest final structures

In [ ]:
from mat3ra.prode import PropertyName

final_structure_properties = client.properties.get_for_job(
    job_id, property_name=PropertyName.non_scalar.final_structure.value
)
visualize([
    {"material": Material.create(client.materials.get(final_structure_property["materialId"])), "title": f"Rank {rank}"}
    for rank, final_structure_property in enumerate(final_structure_properties, start=1)
])

## References

[1] S. Maeda, Y. Harabuchi, M. Takagi, K. Saita, K. Suzuki, T. Ichino, Y. Sumiya, K. Sugiyama and Y. Ono,
"Implementation and performance of the artificial force induced reaction method in the GRRM17 program",
J. Comput. Chem. 39, 233 (2018). https://doi.org/10.1002/jcc.25106

[2] S. Maeda and K. Morokuma, "Communications: A systematic method for locating transition structures of A + B → X
type reactions", J. Chem. Phys. 132, 241102 (2010). https://doi.org/10.1063/1.3457903

[3] K. Sugiyama, Y. Sumiya, M. Takagi, K. Saita and S. Maeda, "Understanding CO oxidation on the Pt(111) surface
based on a reaction route network", Phys. Chem. Chem. Phys. 21, 14366 (2019). https://doi.org/10.1039/c8cp06856a